In [3]:
import os
import glob
import shutil
import threading
import time

In [4]:
def split_file(filepath, chunk_size=int(9.5*1024*1024)):  # 10MB
    """Splits a file into chunks and saves them in a new directory."""

    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    filename = os.path.basename(filepath)
    dirname = filename.split('.')[0] + "_chunks"  # Create directory name
    os.makedirs(dirname, exist_ok=True) # Create directory (or don't error if exists)

    with open(filepath, 'rb') as f:
        chunk_num = 0
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break  # End of file
            chunk_path = os.path.join(dirname, f"{filename}.part{chunk_num}")
            with open(chunk_path, 'wb') as chunk_file:
                chunk_file.write(chunk)
            chunk_num += 1

    print(f"File '{filename}' split into {chunk_num} chunks in directory '{dirname}'.")
    return dirname

In [5]:
def split_file_multithreaded(filepath, chunk_size=int(9.5*1024*1024), num_threads=8):  # Use 4 threads by default
    """Splits a file into chunks using multiple threads."""

    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")

    filename = os.path.basename(filepath)
    dirname = filename.split('.')[0] + "_chunks"
    os.makedirs(dirname, exist_ok=True)

    file_size = os.path.getsize(filepath)
    num_chunks = (file_size + chunk_size - 1) // chunk_size  # Calculate correct number of chunks

    threads = []
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, file_size)  # Calculate end point for each chunk

        def write_chunk(start, end, chunk_num): # Define write_chunk inside loop
             with open(filepath, 'rb') as f:
                 f.seek(start)
                 chunk = f.read(end - start)

                 chunk_path = os.path.join(dirname, f"{filename}.part{chunk_num}")
                 with open(chunk_path, 'wb') as chunk_file:
                    chunk_file.write(chunk)

        thread = threading.Thread(target=write_chunk, args=(start, end, i))
        threads.append(thread)
        thread.start()


        if len(threads) == num_threads or i == num_chunks -1:  # Start threads in batches
            for thread in threads:
                thread.join()
            threads = [] # Clear thread list after joining



    print(f"File '{filename}' split into {num_chunks} chunks in directory '{dirname}'.")

    return dirname

In [6]:
def recombine_file(dirname, output_filename=None, buffer_size=64 * 1024):
    """Recombines file chunks from a directory into a single file."""

    if not os.path.isdir(dirname):
        raise FileNotFoundError(f"Directory not found: {dirname}")

    chunk_files = sorted(glob.glob(os.path.join(dirname, "*.part*"))) # Sort to maintain order

    if output_filename is None:
        output_filename = dirname.split('_chunks')[0]  #  Reconstruct original name if not provided

    # with open(output_filename, 'wb') as outfile:
    #     for chunk_file in chunk_files:
    #         with open(chunk_file, 'rb') as infile:
    #             outfile.write(infile.read())

    with open(output_filename, 'wb') as outfile:
        for chunk_file in chunk_files:
            with open(chunk_file, 'rb') as infile:
                shutil.copyfileobj(infile, outfile, buffer_size)

    print(f"Chunks from '{dirname}' recombined into '{output_filename}'")

    try:
        #shutil.rmtree(dirname)  # Delete the directory and its contents
        print(f"Directory '{dirname}' deleted successfully.")
    except OSError as e:
        print(f"Error deleting directory '{dirname}': {e}")

In [ ]:
# Example usage (replace with your filepath):
filepath = "ffxd.rar"  # Or any other file type
dirname = split_file_multithreaded(filepath)

In [22]:
# Example usage (provide directory name or optionally a new output filename)
# recombine_file(dirname, "recombined_file.bin")  # To specify output filename
#recombine_file(dirname, "ffxd_combined.rar", buffer_size=64 * 1024 *16)

recombine_file("ffxd_chunks", "ffxd_combined.rar", buffer_size= 20 * 1024 * 1024)

Chunks from 'ffxd_chunks' recombined into 'ffxd_combined.rar'
Directory 'ffxd_chunks' deleted successfully.


In [21]:
os.remove("ffxd_combined.rar")

64 x 16 KB = 59s
64x16KB = 1.02m
1*1024*1024 = 57s
20*1024*1024 = 52s